# Layer localization for selective unlearning — Colab runner

Runs the complete experiment on one GPU: prepare the questions, score every layer with three
localization methods, ablate the selected layers, and measure WMDP-Bio against a retain set.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

**Expected runtime on a T4: about 2 hours** for the full grid (3 methods × 8 layer selections ×
5 strengths on 512 test questions, plus development selection and four controls). Step 6 is
resumable — if the session drops, just re-run that cell and it skips everything already saved.
Step 6 also offers a ~25 minute reduced grid if you only want to see the pipeline work.


## 1. Confirm the GPU


In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "T4" not in name:
    print("Note: timings in this notebook assume a T4; a faster GPU will simply finish sooner.")

## 2. Get the code

Either set `REPO_URL` to clone, or run the cell and upload a zip of the repository.

Optionally mount Google Drive first: it keeps `results/` across session drops, so step 7
resumes instead of starting over.


In [ ]:
USE_DRIVE = False   # set True to keep results across disconnects

import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    workdir = pathlib.Path("/content/drive/MyDrive/unlearning_task")
    workdir.mkdir(parents=True, exist_ok=True)
else:
    workdir = pathlib.Path("/content")
os.chdir(workdir)
print("working in", workdir)

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<user>/unlearning_task.git"

if not pathlib.Path("src/unlearning/experiment.py").exists():
    if REPO_URL:
        !git clone --depth 1 $REPO_URL repo
        os.chdir("repo")
    else:
        from google.colab import files
        uploaded = files.upload()               # pick a zip of the repository
        archive = next(iter(uploaded))
        !unzip -q -o "$archive"
        if not pathlib.Path("src/unlearning/experiment.py").exists():
            inner = sorted(pathlib.Path().glob("*/src/unlearning/experiment.py"))
            assert inner, "could not find src/unlearning/experiment.py in the upload"
            os.chdir(inner[0].parents[2])

assert pathlib.Path("src/unlearning/experiment.py").exists(), "repository not found"
os.environ["PYTHONPATH"] = str(pathlib.Path("src").resolve())
print("repository root:", pathlib.Path().resolve())

## 3. Install dependencies

`requirements.txt` holds two verified stacks and pip picks by Python version. The reported run
used transformers 5.16.1 on Python 3.13; Colab's Python resolves to the older pinned stack,
which the code also supports (it handles the `torch_dtype`/`dtype` rename and verifies the
weights really loaded in float32).


In [ ]:
!pip install -q -r requirements.txt
import torch, transformers, pyarrow, matplotlib
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("pyarrow     ", pyarrow.__version__)
print("cuda        ", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 4. Hugging Face access

`meta-llama/Llama-3.2-1B-Instruct` is gated, so a token with access is required. The WMDP and
MMLU data files are public and need no token.


In [ ]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

## 5. Prepare the questions (~1 min)

Downloads the ten pinned Parquet files, drops duplicate questions, and cuts disjoint
localization / development / test splits by SHA-256 rank. Prints the size of each split.


In [ ]:
!python -m unlearning prepare

## 6. Run the experiment

**Full grid, about 2 hours on a T4.** Resumable: re-running skips any condition already saved
under `results/predictions/`, so a dropped session costs only the condition in flight.

Progress lines appear as each stage finishes. If you would rather see the pipeline end to end
in ~25 minutes, use the reduced cell below instead — it keeps both interventions and all four
conditions but tests three strengths instead of five and skips the second gradient objective.


In [ ]:
!python -m unlearning run

In [ ]:
# Reduced alternative (~25 min) — run this *instead of* the cell above, not as well.
# !python -m unlearning run --methods gate,direction --strengths 0,0.5,1.0

## 7. Tables, intervals and figures

Runs on CPU from `results/` alone. Writes the required comparison table per method, every
table the report quotes (`results/report_tables.md`), and the three figures.


In [ ]:
!python -m unlearning report

In [ ]:
import glob
import pandas as pd
for path in sorted(glob.glob("results/table_*.csv")):
    print("=" * 70); print(path)
    display(pd.read_csv(path))

In [ ]:
from IPython.display import Image, display
for name in ("localization.png", "strength.png", "tradeoff.png"):
    display(Image(f"report/figures/{name}"))

In [ ]:
# Headline numbers, straight from the saved analysis.
import json
a = json.load(open("results/analysis.json"))
b = a["baseline"]
print("baseline   WMDP %.2f%%   retain %.2f%%" % (100 * b["forget"]["accuracy"],
                                                  100 * b["retain"]["accuracy"]))
for method in a["methods"]:
    block = a["main"][method]
    print("\n%s  (alpha = %g)" % (method, block["alpha"]))
    for name, interval in block["intervals"]["forget"].items():
        retain = block["intervals"]["retain"][name]
        print("   %-20s dWMDP %+6.2f [%+6.2f, %+6.2f]   dRetain %+6.2f [%+6.2f, %+6.2f]" % (
            name, interval["drop_pp"], *interval["interval_95"],
            retain["drop_pp"], *retain["interval_95"]))
    stability = a["localization"][method]["split_half_spearman"]
    print("   split-half stability:", {k: round(v, 3) for k, v in stability.items()})

## 8. Save the results

Downloads everything needed to reproduce the report offline: per-question predictions, layer
scores, bootstrap output, tables and figures.


In [ ]:
!tar czf results.tgz results report data
from google.colab import files
files.download("results.tgz")